In [1]:
import os

In [2]:
os.chdir("../")

In [3]:
%pwd

'd:\\DataScience\\Wine-Quality-MLOps-Pipeline'

In [4]:
from dataclasses import dataclass
from pathlib import Path

@dataclass(frozen=True)
class DataTransformationConfig:
    root_dir: Path
    data_path: Path
    STATUS_FILE: Path
    train_data_path: Path
    test_data_path: Path



In [5]:
from mlProject.utils.common import read_yaml, create_directories
from mlProject.constants import *

In [6]:
class ConfigurationManager:
    
    def __init__(self, config_filepath=CONFIG_FILE_PATH, params_filepath=PARAMS_FILE_PATH, schema_filepath=SCHEMA_FILE_PATH):
        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)
        self.schema = read_yaml(schema_filepath)

        create_directories([self.config.artifacts_root])
    
    def get_data_transformation_config(self) -> DataTransformationConfig:
        config = self.config.data_transformation
        
        create_directories([config.root_dir])

        return DataTransformationConfig(
            root_dir = Path(config.root_dir),
            data_path = Path(config.data_path),
            STATUS_FILE = Path(config.STATUS_FILE),
            train_data_path = Path(config.train_data_path),
            test_data_path = Path(config.test_data_path)
        )

In [7]:
import pandas as pd
from sklearn.model_selection import train_test_split
from mlProject import logger

In [8]:
class DataTransformation:
    def __init__(self, config: DataTransformationConfig):
        self.config = config
        self.data = pd.read_csv(self.config.data_path)


    def handle_missing_and_duplicates(self) -> pd.DataFrame:

        with open(self.config.STATUS_FILE , "r") as fh :
            report = fh.read()

#---Missing Values---

        missing_line = next(
            (line for line in report.splitlines() if line.startswith("Missing values by column:")),""
        )

        has_missing = "None" not in missing_line

        if has_missing:
            before_rows = len(self.data)
            self.data = self.data.dropna()
            dropped_rows = before_rows - len(self.data)
            logger.info(f"Dropped {dropped_rows} rows containing missing values")
            logger.info(f"Number of rows after handling missing values : {len(self.data)}")
        else:
            logger.info("No missing values found!")


#---Duplicate Values---

        dup_line = next(
        (line for line in report.splitlines() if line.startswith("Duplicate rows:")),
        "Duplicate rows: 0"
       )
        reported_duplicates = int(dup_line.split(":")[1].strip())

        if reported_duplicates > 0:
            before = len(self.data)
            self.data = self.data.drop_duplicates()
            after = len(self.data)
            logger.info(f"Dropped {before - after} duplicate rows!")
            logger.info(f"Final number of rows : {after}")

        else:
            logger.info("No duplicate rows found!")


        return self.data
    
    def train_test_splitting(self):
        """
        Stratified on binned quality so rare extreme-quality wines
        (e.g. quality < 3 or > 7) aren't left out of either split entirely.
        """
        bins = pd.cut(self.data["quality"], bins=[0, 4, 6, 10], labels=["low", "mid", "high"])
 
        train, test = train_test_split(
            self.data,
            test_size=0.2,
            random_state=42,
            stratify=bins,
        )
 
        train.to_csv(self.config.train_data_path, index=False)
        test.to_csv(self.config.test_data_path, index=False)
 
        logger.info(f"Train shape: {train.shape}")
        logger.info(f"Test shape: {test.shape}")
    
    
    def run_transformation(self):
        self.handle_missing_and_duplicates()
        self.train_test_splitting()

In [9]:
try:
    config = ConfigurationManager()
    data_transformation_config = config.get_data_transformation_config()
    data_transformation = DataTransformation(config=data_transformation_config)
    status = data_transformation.run_transformation()
except Exception as e:
    raise e

[2026-07-21 17:47:12,217: INFO: common]: yaml file: config\config.yaml loaded successfully
[2026-07-21 17:47:12,221: INFO: common]: yaml file: params.yaml loaded successfully
[2026-07-21 17:47:12,227: INFO: common]: yaml file: schema.yaml loaded successfully
[2026-07-21 17:47:12,229: INFO: common]: created directory at: artifacts
[2026-07-21 17:47:12,234: INFO: common]: created directory at: artifacts/data_transformation


[2026-07-21 17:47:12,248: INFO: 3850062548]: No missing values found!
[2026-07-21 17:47:12,256: INFO: 3850062548]: Dropped 240 duplicate rows!
[2026-07-21 17:47:12,258: INFO: 3850062548]: Final number of rows : 1359
[2026-07-21 17:47:12,288: INFO: 3850062548]: Train shape: (1087, 12)
[2026-07-21 17:47:12,290: INFO: 3850062548]: Test shape: (272, 12)
